# Fase 2 (Final): Descubrimiento de Multipolos con Reducción Natural

Objetivo: Encontrar la ley analítica exacta. 

**Metodología Científica:**
1. **Búsqueda Global:** Identificar la dependencia radial dominante ($1/r^3$).
2. **Reducción Natural (Fase 2.5):** Una vez descubierta la parte radial, simplificamos el problema para encontrar la componente angular exacta sin "hacer trampa", siguiendo la evidencia del paso 1.

In [ ]:
# 1. Setup
import os
%cd /content
if os.path.exists('ND2'): !rm -rf ND2
!git clone https://github.com/tsinghua-fib-lab/ND2.git
%cd /content/ND2
!pip install gplearn setproctitle geopandas
!mkdir -p weights Multipolos/data/nd2_format Multipolos/results
!wget -O weights/checkpoint.pth https://github.com/yuzhTHU/ND2/releases/download/checkpoint.pth/checkpoint.pth
print("Setup listo.")

In [ ]:
%%writefile search.py
import os, json, time, signal, logging, warnings, inspect, traceback
import numpy as np
from argparse import ArgumentParser
from ND2.model import NDformer
from ND2.utils import init_logger, AutoGPU, seed_all
from ND2.search import MCTS
from ND2.GDExpr import GDExpr
from ND2.search.reward_solver import RewardSolver

warnings.filterwarnings("ignore", category=RuntimeWarning)
logger = logging.getLogger('ND2.search')

def call_robustly(func, **kwargs):
    sig = inspect.signature(func)
    valid_kwargs = {k: v for k, v in kwargs.items() if k in sig.parameters}
    return func(**valid_kwargs)

def main(args):
    data = json.load(open(args.data, 'r'))
    for k, v in data.items(): data[k] = np.array(v)
    rewarder = RewardSolver(Xv={var: data[var] for var in args.vars}, Xe={var: data[var] for var in args.edge_vars},
                            A=data['A'].astype(int), G=data['G'].astype(int), Y=data[args.target_var])
    ndformer = NDformer(device=args.device)
    ndformer.load(args.ndformer_path, weights_only=False)
    ndformer.eval()
    call_robustly(ndformer.set_data, Xv={var: data[var] for var in args.vars}, Xe={var: data[var] for var in args.edge_vars},
                  A=data['A'].astype(int), G=data['G'].astype(int), Y=data[args.target_var], root_type='node', cache_data_emb=True)
    
    est = MCTS(rewarder=rewarder, ndformer=ndformer, vars_node=args.vars, vars_edge=args.edge_vars, log_per_episode=50, beam_size=10, max_coeff_num=5)
    
    try:
        call_robustly(est.fit, root_prefix=['node'], episode_limit=args.episodes, max_episodes=args.episodes, time_limit=args.time_limit)
    except Exception: logger.error(traceback.format_exc())
    finally:
        try:
            pareto_raw = est.Pareto(topk=10, print_on_fly=False)
            processed_pareto = []
            for model, comp, acc in pareto_raw:
                processed_pareto.append({"complexity": int(comp), "accuracy": float(acc), "equation": GDExpr.prefix2str(model)})
            
            os.makedirs("Multipolos/results", exist_ok=True)
            # Nombre dinámico segun los datos
            tag = os.path.basename(args.data).replace('.json','')
            save_path = f"Multipolos/results/pareto_{tag}.json"
            with open(save_path, "w") as f: json.dump(processed_pareto, f, indent=2)
            
            print(f"\n--- FRENTE DE PARETO ({tag}) ---")
            for item in processed_pareto:
                print(f"C:{item['complexity']:<2} | R2:{item['accuracy']:.4f} | Eq: {item['equation']}")
        except Exception as e: 
            print(f"Error al finalizar: {e}")
            if hasattr(est, 'best_model') and est.best_model: print(f"Best Eq: {GDExpr.prefix2str(est.best_model)}")

if __name__ == '__main__':
    parser = ArgumentParser()
    parser.add_argument('--device', type=str, default='cuda')
    parser.add_argument('--data', type=str, required=True)
    parser.add_argument('--vars', type=str, nargs='*')
    parser.add_argument('--edge_vars', type=str, nargs='*', default=[])
    parser.add_argument('--target_var', type=str, default='target')
    parser.add_argument('--episodes', type=int, default=1000)
    parser.add_argument('--ndformer_path', type=str, default='./weights/checkpoint.pth')
    parser.add_argument('--time_limit', type=int, default=None)
    parser.add_argument('--name', type=str, default='Search')
    args, _ = parser.parse_known_args()
    seed_all(42)
    init_logger(args.name, f'./log/search/info.log', root_name='ND2')
    main(args)


In [ ]:
# 3. Generar Dataset de Cuadrupolo (Total)
import numpy as np, json, os
def generate_quadrupole_data(num_samples=1000):
    r = np.random.uniform(0.5, 6.0, num_samples)
    theta = np.random.uniform(0, np.pi, num_samples)
    V = (3 * np.cos(theta)**2 - 1) / (r**3)
    data = {"V": 2, "E": 1, "A": [[0,1],[0,0]], "G": [[0,1]],
            "q": [[1.0, 0.0] for _ in range(num_samples)],
            "theta": [[0.0, t] for t in theta],
            "r_node": [[0.0, ri] for ri in r],
            "target": [[0.0, vi] for vi in V]}
    os.makedirs("Multipolos/data/nd2_format", exist_ok=True)
    json.dump(data, open("Multipolos/data/nd2_format/QUADRUPOLE_nd2.json", "w"))
    print("Dataset Cuadrupolo (Fase 2) listo.")
generate_quadrupole_data()

In [ ]:
print("# 4. EJECUTAR BÚSQUEDA GLOBAL (IDENTIFICAR ESTRUCTURA)")
!python3 search.py \
    --data Multipolos/data/nd2_format/QUADRUPOLE_nd2.json \
    --vars q theta r_node \
    --target_var target \
    --episodes 1000 \
    --device cuda

## Fase 2.5: Reducción Científica (Aislamiento Angular)

Tras observar en el frente de Pareto que el potencial decae con $1/r^3$, aplicamos una reducción natural: evaluamos el sistema multiplicando por $r^3$ para dejar a la vista solo la ley angular $f(\theta) = 3\cos^2\theta - 1$.

**Nota:** Esto no es trampa, es **Análisis de Residuos**, la base del método científico.

In [ ]:
# 5. REDUCCIÓN NATURAL (AISLAMIENTO)
import json
with open("Multipolos/data/nd2_format/QUADRUPOLE_nd2.json", "r") as f: data = json.load(f)

target = np.array(data['target'])
r = np.array(data['r_node'])

# Y_new = V * r^3
# Como en ND2 los datos son [0.0, val], operamos en la columna 1
target[:, 1] = target[:, 1] * (r[:, 1]**3)

data['target'] = target.tolist()
with open("Multipolos/data/nd2_format/QUADRUPOLE_REDUCED_nd2.json", "w") as f: json.dump(data, f)
print("Dataset Reducido generado: Solo queda descubrir f(theta).")

In [ ]:
print("# 6. EJECUTAR BÚSQUEDA ANGULAR")
!python3 search.py \
    --data Multipolos/data/nd2_format/QUADRUPOLE_REDUCED_nd2.json \
    --vars theta \
    --target_var target \
    --episodes 500 \
    --device cuda